In [2]:
# Importing
import os
import h5py
import json
import numpy as np
import matplotlib.pyplot as plt
import scipy.signal as sp
import xdf
import psyphyea
import glob
import csv
import math
import warnings
import pandas as pd
plt.style.use('fivethirtyeight')
warnings.filterwarnings("ignore")
features = ['EmpaticaAcceleration','EmpaticaBVP','EmpaticaGSR','EmpaticaTemperature']

In [5]:
subject = [21,24,26,27,28,29,30,31,32,33,34,35,36,37,38]
#sessions = [[2,3,4,5,6],[3,4,5,6],[1,2,3,5,6],[4,6],[1,2,3,4,6],[1,2,3,4,5,6],[1,3,4,5,6],[1,2,3,4,5,6],[1,2,3,4,5,6],
            #[1,2,3,4,5,6],[2,3,4,5,6],[2,3,5],[1,2,3,5,6],[1,2,3,4,6],[1,2,3],[1,2,3,4]]

In [6]:

def get_file_path(sub_id):
    file_dir = '/mnt/neurodata/NeurofeedbackStudy/data/processed/0' + str(sub_id)   # Yields good result
    #file_dir = '/mnt/neurodata/NeurofeedbackStudy/data/processed/029'
    h5_filepaths = glob.glob(file_dir + '/*')
    print(*h5_filepaths,sep = '\n')
    filelist = []
    for i in range(len(h5_filepaths)):
        if h5_filepaths[i].find('SHL.') != -1 or h5_filepaths[i].find('SLH.') != -1:
            filelist.append(i)
    temp = [0 ] * 6

    for i in filelist:
        st = h5_filepaths[i]
        indx = int(st[st.rfind('/') + 5:st.find('.') - 3])
        temp[indx-1] = int(i)

    filelist = temp
    print (filelist)
    return h5_filepaths,filelist


def find_gun_pose(data,time,target_time,dims): # Takes long time
    out = []
    index_match = []
    # dimensions spectified here
    #dims = 3
    l = len(data)
    for i in target_time:
        i_array = np.repeat(i,l)
        diff = time - i_array
        matched_index = np.argmin(abs(diff))
        out.append(data[matched_index,:dims])
        index_match.append(matched_index)
    return out,index_match

def euclidean_distance(a,b):
    import math
    a = np.array(a)
    b = np.array(b)
    distance = a - b
    distance = distance * distance
    return math.sqrt(sum(distance))


def load_data(h5_filepaths,file_select,ch_name):
    # Creating streams from h5 file
    streams = psyphyea.StreamContainer.read_from_h5(h5_filepaths[file_select])
    # Showing extracted streams
    streams.get_as_dataframe()
    # Generating shooting table
    target_table,shot_table = psyphyea.generate_shooting_tables(streams)
    # Creating h5 data handler
    h5 = h5py.File(h5_filepaths[file_select],'r')
    def func(y,x):print('Name: %40s Attrs: %d'%(x.name,len(x.attrs)))
    #h5.visititems(func)
    # Picking out GunPose data
    attributes = json.loads(h5[ch_name].attrs['attrs_as_json'])
    # Getting the column names
    labels = [x['label'][0] for x in attributes['desc']['channels'][0]['channel']]
    #print(labels)
    gun_pose = np.array(h5[ch_name + '/time_series'])
    gun_time = np.array(h5[ch_name + '/time_stamps'])

    return target_table,gun_pose,gun_time



def data_pick_from_h5(h5_filepaths,file_select,ch_name,dims):
    target_table,gun_pose,gun_time = load_data(h5_filepaths,file_select,ch_name)
    print('Channel name = {} Data length = {}'.format(ch_name,len(gun_time)))
    target_pop_time = np.array(target_table['time_stamps'])  # Using old target table because we need initial gun position for all targets
  
    # Finding the target fade for the targets   
    target_fade_time = np.array(target_table['time_stamps'] + target_table['lifetime'])

        # Extracting initial gun position
        # Initial means data taken at target pop time
    initial_gun_pose,initial_index_match = find_gun_pose(gun_pose,gun_time,target_pop_time,dims)
    final_gun_pose,final_index_match = find_gun_pose(gun_pose,gun_time,target_fade_time,dims)


        # Extracting the target_id for misses
    target_id_miss = np.array(target_table.index[target_table.num_shots_to_hit == -1]/2)
        # Extracting the target_id for hits
    target_id_hit = np.array(target_table.index[target_table.num_shots_to_hit == 1]/2)

        # ins_velocity_miss = []
    duration = []  # Used later. To extract the duration of data
        # stress_miss = []
    l = len(initial_index_match)
    data = []
    for st,ed in zip(initial_index_match,final_index_match):
        start = st
        end = ed
        position_temp = gun_pose[start:end + 1,0:dims] # Adding 1 because we want the end point to be included also
        time_temp = gun_time[start:end + 1] # Adding 1 because we want the end point to be included also
        duration.append(time_temp)

        data.append(position_temp.transpose())
        
    return data,duration

def read_data(filename, half):
    raw_data = []
    time = []
    with open(filename,'r') as fi:
        for i in fi:
            if i.find('time') == -1:
                i = i.split(',') # Splitting the data
                #print(i)
                i = [float(x) for x in i] # Converting the strings into floats
                length = len(i)
                if half == True:
                    raw_data.append(i[:int(length/2)])
                else:
                    raw_data.append(i[-1]) # Splitting the label (Last element of each rows)
                time.append(i[0]) # Appending the labels after converting it to integers
            
    return np.array(raw_data),np.array(time)

In [72]:
sub_id = 33 # Change subject ID to extract data
h5_filepaths,filelist = get_file_path(sub_id)

/mnt/neurodata/NeurofeedbackStudy/data/processed/033/033S2NFB.h5
/mnt/neurodata/NeurofeedbackStudy/data/processed/033/033S2SLH.h5
/mnt/neurodata/NeurofeedbackStudy/data/processed/033/033S3NFB.h5
/mnt/neurodata/NeurofeedbackStudy/data/processed/033/033S5SHL.h5
/mnt/neurodata/NeurofeedbackStudy/data/processed/033/033S1NFB.h5
/mnt/neurodata/NeurofeedbackStudy/data/processed/033/033S3SHL.h5
/mnt/neurodata/NeurofeedbackStudy/data/processed/033/033S6SLH.h5
/mnt/neurodata/NeurofeedbackStudy/data/processed/033/033S0THR.h5
/mnt/neurodata/NeurofeedbackStudy/data/processed/033/033S4NFB.h5
/mnt/neurodata/NeurofeedbackStudy/data/processed/033/033S1SHL.h5
/mnt/neurodata/NeurofeedbackStudy/data/processed/033/033S4SLH.h5
/mnt/neurodata/NeurofeedbackStudy/data/processed/033/033S5NFB.h5
[9, 1, 5, 10, 3, 6]


In [73]:
session_index = 1
for file_select in filelist:
    
    target_table,gun_pose,gun_time = load_data(h5_filepaths,file_select,ch_name = 'GunPose')
    target_pop_time = np.array(target_table['time_stamps'])  # Using old target table because we need initial gun position for all targets
    target_num = len(target_pop_time)
    
    # Finding the target fade for the targets   
    target_fade_time = np.array(target_table['time_stamps'] + target_table['lifetime'])
    
    # Extracting initial gun position
    # Initial means data taken at target pop time
    initial_gun_pose,initial_index_match = find_gun_pose(gun_pose,gun_time,target_pop_time,dims = 3)
    final_gun_pose,final_index_match = find_gun_pose(gun_pose,gun_time,target_fade_time,dims = 3)
    

    # Extracting the target_id for misses
    target_id_miss = np.array(target_table.index[target_table.num_shots_to_hit == -1]/2)
    # Extracting the target_id for hits
    target_id_hit = np.array(target_table.index[target_table.num_shots_to_hit == 1]/2)

    # ins_velocity_miss = []
    duration = []  # Used later. To extract the duration of data
    # stress_miss = []
    l = len(initial_index_match)
    ins_v = []
    for st,ed in zip(initial_index_match,final_index_match):
        start = st
        end = ed
        position_temp = gun_pose[start:end + 1,0:3] # Adding 1 because we want the end point to be included also
        time_temp = gun_time[start:end + 1] # Adding 1 because we want the end point to be included also
        duration.append(time_temp)

        velocity_temp = []

        for j in range(len(time_temp) - 1):  # -1 because we are ommiting the last data point
            delta_t = time_temp[j+1] - time_temp[j]
            delta_s  = np.sqrt(np.sum((position_temp[j+1]-position_temp[j])**2)) #l2 dist
            #velocity_temp = delta_s/delta_t
            velocity_temp.append(delta_s/(delta_t))

        ins_v.append(velocity_temp)

    # Getting stress level
    stress = np.array(target_table['high_stress'])

    # Getting target hit\ miss
    label = np.array(target_table['num_shots_to_hit'])
    miss_indx = np.arange(len(label))[label == -1]
    label[miss_indx] = 0

    with h5py.File('data' + str(sub_id) + '.h5','a') as fi:
        dt = h5py.special_dtype(vlen=np.dtype('float64'))
        
        if 'Gun_velocity/Session_' + str(session_index) + '/label' in fi:
            del fi['Gun_velocity/Session_' + str(session_index) + '/label']
            del fi['Gun_velocity/Session_' + str(session_index) + '/stress']
            del fi['Gun_velocity/Session_' + str(session_index) + '/data']
        
        dset = fi.create_dataset('Gun_velocity/Session_' + str(session_index) + '/label',(target_num,),dtype = 'i',data = label)
        dset = fi.create_dataset('Gun_velocity/Session_' + str(session_index) + '/stress',(target_num,),dtype = 'i',data = stress)
        dset = fi.create_dataset('Gun_velocity/Session_' + str(session_index) + '/data',(target_num,),dtype = dt)

        for i,x in enumerate(ins_v):
            dset[i] = np.array(x).flatten()

    print('Session {} data written'.format(session_index))
        
    session_index += 1

print('{} file written'.format('data' + str(sub_id) + '.h5','a'))

ValueError: cannot set a frame with no defined index and a scalar

In [ ]:
session_index = 1
for file_select in filelist:
    for ch_name in features:
        
        if ch_name == 'EmpaticaAcceleration':
            data,duration = data_pick_from_h5(h5_filepaths,file_select,ch_name,dims = 3)
        elif ch_name == 'EmpaticaHR':
            data,duration = data_pick_from_h5(h5_filepaths,file_select,ch_name,dims = 2)
        else:
            data,duration = data_pick_from_h5(h5_filepaths,file_select,ch_name,dims = 1)
        
        data_len = len(data)
        
        with h5py.File('data' + str(sub_id) + '.h5','a') as fa:
            dt = h5py.special_dtype(vlen=np.dtype('float32'))
            #dset = fa.create_dataset( str(ch_name) + '/Session_' + str(session_index) + '/label',(720,),dtype = 'i',data = label)
            #dset = fi.create_dataset('Gun_veclocity/Session_' + str(session_index) + '/stress',(720,),dtype = 'i',data = label_stress)
            
            if str(ch_name) + '/Session_' + str(session_index) + '/data' in fa:
                del fa[str(ch_name) + '/Session_' + str(session_index) + '/data']
            
            elif ch_name == 'EmpaticaAcceleration':
                dset = fa.create_dataset(str(ch_name) + '/Session_' + str(session_index) + '/data',(data_len,3,),dtype = dt)
                for j in range(3):
                    for i,x in enumerate(data):
                        dset[i,j] = x[j]
            elif ch_name == 'EmpaticaHR':
                dset = fa.create_dataset(str(ch_name) + '/Session_' + str(session_index) + '/data',(data_len,2,),dtype = dt)
                for j in range(2):
                    for i,x in enumerate(data):
                        dset[i,j] = x[j]
            else:
                dset = fa.create_dataset(str(ch_name) + '/Session_' + str(session_index) + '/data',(data_len,),dtype = dt)
                for i,x in enumerate(data):
                    dset[i] = x.flatten()
                    
    print('Session {} data written'.format(session_index))
        
    session_index += 1
    
print('{} file wriiten'.format('data' + str(sub_id) + '.h5'))

In [ ]:
session_index = 1
ch_name = 'EmpaticaHR'
for file_select in filelist:
    streams = psyphyea.StreamContainer.read_from_h5(h5_filepaths[file_select])
    # Showing extracted streams
    streams.get_as_dataframe()
    # Generating shooting table
    target_table,shot_table = psyphyea.generate_shooting_tables(streams)
    
    target_pop_time = np.array(target_table['time_stamps'])  # Using old target table because we need initial gun position for all targets

    # Finding the target fade for the targets   
    target_fade_time = np.array(target_table['time_stamps'] + target_table['lifetime'])

    data_len = len(target_pop_time)
    
    # Extracting initial gun position
    # Initial means data taken at target pop time
    st = h5_filepaths[file_select]
    st = st[st.rfind('/') + 1:st.find('.')]
    file = './hrdata/'+ st +'_hr_df.csv'
    gun_pose,gun_time = read_data(file,half = False)
    #initial_gun_pose,initial_index_match = find_gun_pose(gun_pose,gun_time,target_pop_time,1)
    print('Channel name = {} Data length = {}'.format(ch_name,len(gun_time)))
    initial_gun_pose = []
    initial_index_match = []
    final_gun_pose = []
    final_index_match = []
    # dimensions spectified here
    #dims = 3
    l = len(gun_pose)
    for i in target_pop_time:
        i_array = np.repeat(i,l)
        diff = gun_time - i_array
        matched_index = np.argmin(abs(diff))
        initial_gun_pose.append(gun_pose[matched_index])
        initial_index_match.append(matched_index)
        
    for i in target_fade_time:
        i_array = np.repeat(i,l)
        diff = gun_time - i_array
        matched_index = np.argmin(abs(diff))
        final_gun_pose.append(gun_pose[matched_index])
        final_index_match.append(matched_index)


            # Extracting the target_id for misses
    target_id_miss = np.array(target_table.index[target_table.num_shots_to_hit == -1]/2)
            # Extracting the target_id for hits
    target_id_hit = np.array(target_table.index[target_table.num_shots_to_hit == 1]/2)

            # ins_velocity_miss = []
    duration = []  # Used later. To extract the duration of data
            # stress_miss = []
    l = len(initial_index_match)
    data = []
    for st,ed in zip(initial_index_match,final_index_match):
        start = st
        end = ed
        position_temp = gun_pose[start:end + 1] # Adding 1 because we want the end point to be included also
        time_temp = gun_time[start:end + 1] # Adding 1 because we want the end point to be included also
        duration.append(time_temp)

        data.append(position_temp.transpose())

    with h5py.File('data' + str(sub_id) + '.h5','a') as fa:
        dt = h5py.special_dtype(vlen=np.dtype('float32'))
        if str(ch_name) + '/Session_' + str(session_index) + '/data' in fa:
            del fa[str(ch_name) + '/Session_' + str(session_index) + '/data']
        dset = fa.create_dataset(str(ch_name) + '/Session_' + str(session_index) + '/data',(data_len,),dtype = dt)
        for i,x in enumerate(data):
            dset[i] = x.flatten()


    print('Session {} data written'.format(session_index))
    
    session_index += 1
print('{} file wriiten'.format('data' + str(sub_id) + '.h5'))